# **Mini-Projeto Avaliativo para o curso de Visualização de Dados e BI do programa SCTEC.**

Entregar um script em Python que realize uma Análise Exploratória da base Varejo seguindo etapas claras, documentadas e reproduzíveis.

Etapas obrigatórias:
* Carregar a base Varejo.csv com pandas e mostrar: número de registros, colunas e tipos de dados.
* Verificar e reportar ao menos dois problemas básicos: valores nulos por coluna, duplicatas e possíveis inconsistências (ex.: datas inválidas ou categorias vazias).
* Fazer as três etapas de limpeza mínima necessária: remover ou imputar nulos (explique a escolha), eliminar duplicatas relevantes e ajustar tipos de dados (ex.: converter coluna DATA para datetime).
* Gerar estatísticas descritivas básicas para coluna de número de filhos do cliente (média; mediana; desvio padrão; moda; máximo; mínimo; e contagem).
* Explorar padrões de agrupamento com pelo menos dois agrupamentos (por exemplo: gênero com mais vendas, compras), usando groupby() ou pivot_table().
* Produzir um pequeno bloco de conclusões (3–6 tópicos) com os principais insights obtidos e possíveis problemas remanescentes na base.


In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("/kaggle/input/datasets/namespaiva/base-varejo/Base Varejo.csv", delimiter=";")

df

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
829995,19/08/2022,919822,155,F,2,0,B,183,ALIMENTOS,KETCHUP,NaN,NaN,NaN,NaN
829996,19/08/2022,919822,155,F,2,0,B,56,ALIMENTOS,QUEIJO MUSSARELA,NaN,NaN,NaN,NaN
829997,19/08/2022,919822,155,F,2,0,B,227,ALIMENTOS,ARROZ,NaN,NaN,NaN,NaN
829998,19/08/2022,919822,155,F,2,0,B,214,ALIMENTOS,CEBOLA,NaN,NaN,NaN,NaN


# Análise rápida 
O Dataframe não tem um ID/chave única

As últimas 4 colunas visivelmente tem dados vazios. 

Lista de colunas retirada da documentação do dataset:
* DATA: Data da compra
* CO_ID: Identificação do número de compra (número da nota fiscal)
* CL_ID: Identificação do cliente (número do cliente)
* CL_GENERO: Sexo biológico informado pelo cliente
* CL_EC: Estado civil do cliente (Casado ou união estával, Divorciado, Separado, Solteiro, Viúvo)
* CL_FHL: Número de filhos do cliente
* CL_SEG: Segmentação econômica do cliente (classe A, B ou C)
* PR_ID: Código do produto (SKU) adquirido
* PR_CAT: Categoria do produto adquirido
* PR_NOME: Nome do produto adquirido

In [2]:
df.info()
# A base está com dados nulos nas últimas 4 colunas.
# Total de 830.000 registros.
# O campo data pode ser convertido para tipo Datetime, facilitando a sua posterior manipulação.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  object 
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  object 
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  object 
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  object 
 9   PR_NOME      830000 non-null  object 
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), object(5)
memory usage: 88.7+ MB


In [3]:
# Excluir as últimas 4 colunas vazias
df = df.drop(columns=['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13'])

In [4]:
df.isnull().sum()
# Verificando se além das 4 últimas colunas, existem outras com campos vazios. 
# Confirmado que não constam dados vazios.

DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64

# Atualizando os tipos de colunas correspondentes aos dados das colunas
Neste dataset será transformada a coluna DATA que chegou como tipo object e será modificada para o tipo datetime. Desta forma é possível manipular o campo para montar relatórios estratégicos de vendas

In [5]:
# Passa para string, remove espaços vazios e força a conversão novamente para o tipo string
df['DATA'] = df['DATA'].astype(str).str.strip()

# Conversão da coluna para o tipo datatime
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y', errors='coerce')

df

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,2019-02-01,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,2019-02-01,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,2019-02-01,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,2019-02-01,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,2019-02-01,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO
...,...,...,...,...,...,...,...,...,...,...
829995,2022-08-19,919822,155,F,2,0,B,183,ALIMENTOS,KETCHUP
829996,2022-08-19,919822,155,F,2,0,B,56,ALIMENTOS,QUEIJO MUSSARELA
829997,2022-08-19,919822,155,F,2,0,B,227,ALIMENTOS,ARROZ
829998,2022-08-19,919822,155,F,2,0,B,214,ALIMENTOS,CEBOLA


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   DATA       830000 non-null  datetime64[ns]
 1   CO_ID      830000 non-null  int64         
 2   CL_ID      830000 non-null  int64         
 3   CL_GENERO  830000 non-null  object        
 4   CL_EC      830000 non-null  int64         
 5   CL_FHL     830000 non-null  int64         
 6   CL_SEG     830000 non-null  object        
 7   PR_ID      830000 non-null  int64         
 8   PR_CAT     830000 non-null  object        
 9   PR_NOME    830000 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(4)
memory usage: 63.3+ MB


# Verificando e tratando linhas com dados duplicados

In [7]:
df.duplicated().sum()
# A base possui 96.553 registros duplicados. 

np.int64(96553)

In [8]:
# Excluir os registros duplicados
df = df.drop_duplicates()

In [9]:
df.info() # 733.447 registros

<class 'pandas.core.frame.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   DATA       733447 non-null  datetime64[ns]
 1   CO_ID      733447 non-null  int64         
 2   CL_ID      733447 non-null  int64         
 3   CL_GENERO  733447 non-null  object        
 4   CL_EC      733447 non-null  int64         
 5   CL_FHL     733447 non-null  int64         
 6   CL_SEG     733447 non-null  object        
 7   PR_ID      733447 non-null  int64         
 8   PR_CAT     733447 non-null  object        
 9   PR_NOME    733447 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(4)
memory usage: 61.6+ MB


# Verificando se os campos do tipo string, tem algum valor fora do padrão e que precise ser padronizado

In [10]:
df['CL_GENERO'].value_counts()
# Já podemos perceber que tem quase a mesma quantidade de clientes do sexo feminino como do masculino
# O feminino ultrapassa um pouco mais da metade

CL_GENERO
F    382427
M    351020
Name: count, dtype: int64

In [11]:
df['CL_SEG'].value_counts()
# Percebemos que o maior público é de clientes da classe B

CL_SEG
B    468505
C    205265
A     59677
Name: count, dtype: int64

In [12]:
df['PR_CAT'].value_counts()
# Quase que a maioria dos produtos vendidos são da categoria de Alimentos

PR_CAT
ALIMENTOS     384197
HIGIENE       137702
LIMPEZA       128632
BEBIDAS        38264
PET            28553
ACESSORIOS     12871
#N/D            3228
Name: count, dtype: int64

In [13]:
# Pesquisado para ter uma visão de como está a escrita dos produtos, por padrão ordena por quantidade
df['PR_NOME'].value_counts()
# Vemos que o Presunto é o produto que tem maior venda. Quase o dobro comparando com os produtos mais vendidos na sequência.

PR_NOME
PRESUNTO COZIDO          12719
SARDINHA                  6610
ESCOVA DE DENTE           6518
BANANA                    6518
GEL                       6517
                         ...  
#N/D                      3228
ABSORVENTE                3220
ALIMENTO PARA PASSARO     3198
ALHO                      3194
ALCOOL                    3168
Name: count, Length: 118, dtype: int64

In [14]:
# Padronizar os dados do tipo string para retirar espaços em branco (no inio e fim) e, neste caso, manter todo em caixa alta. 
# Consultei fórmula por IA, via Google
colunas_string = df.select_dtypes(include=['object']).columns
#df[colunas_string] = df[colunas_string].apply(lambda x: x.astype(object).str.strip())

# Use o .loc com ":" para indicar "todas as linhas" e passe a lista de colunas
df.loc[:, colunas_string] = df[colunas_string].apply(lambda x: x.astype(str).str.strip())


In [15]:
df[colunas_string]

,CL_GENERO,CL_SEG,PR_CAT,PR_NOME
0,M,C,BEBIDAS,REFRIGERANTE GUARANA
1,M,C,BEBIDAS,REFRIGERANTE OUTROS
2,M,C,HIGIENE,LENCO UMEDECIDO
3,M,C,ALIMENTOS,ABACAXI
4,M,C,LIMPEZA,LIMPADOR MULTIUSO
...,...,...,...,...
829991,F,B,HIGIENE,PRESERVATIVO
829993,F,B,ALIMENTOS,SNACKS
829994,F,B,ALIMENTOS,AZEITE
829998,F,B,ALIMENTOS,CEBOLA
